## 0. User Configuration

**Edit the cell below before running anything else.**
Set the paths to your CSV gene list and H5AD atlas, the relevant column names, and the output directory.
All downstream cells use these variables automatically.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# USER CONFIGURATION — edit the values in this cell, then run all cells
# ═══════════════════════════════════════════════════════════════════════════

# ── Disease gene list CSV ───────────────────────────────────────────────────
GENE_CSV_PATH   = "/home/kibr/Work/Vas_try/Data/brain_vascular_disease_genes_curated.csv"   # ← set your path
DISEASE_COL     = "condition"     # ← column name for disease labels
GENE_COL        = "gene" #"gene_name"  # ← column name for gene symbols

# ── H5AD single-cell atlas ──────────────────────────────────────────────────
H5AD_PATH       = "/home/kibr/Work/Vascular_atlas/Results/Revision_2/clean_object_revision_03032026.h5ad"              # ← set your path
CELL_CLASS_COL  = "Cell_class"  # ← obs column for cell type labels
DONOR_COL       = "individualID"# ← obs column for donor / sample ID

# ── Output directory ────────────────────────────────────────────────────────
OUT             = "/home/kibr/Work/Vas_try/Results"              # ← set your path

# ── Analysis parameters (safe to leave as-is) ──────────────────────────────
N_BOOT          = 10_000   # bootstrap iterations
SEED            = 42       # random seed
CHUNK           = 2000     # gene-chunk size for CPM computation

# ── Vascular cell types to highlight (adjust to match your atlas) ──────────
VASC_TYPES      = ["Endothelial", "Mural_Cell", "Fibroblast"]

print("Configuration loaded.")
print(f"  Gene CSV  : {GENE_CSV_PATH}")
print(f"  H5AD      : {H5AD_PATH}")
print(f"  Output dir: {OUT}")


# Step 2 — EWCE Bootstrap Enrichment Analysis

### Analysis steps
0. User configuration (paths, columns, parameters)
1. Imports & load gene list CSV
2. Optional: add / remove genes
3. Colour palettes
4. Initialise config variables
5. Load AnnData (H5AD)
6. Pre-compute library sizes
7. Donor-balanced CPM means
8. Specificity matrix
9. Gene–atlas overlap check
10. EWCE bootstrap
11. FDR correction & visualisations
12. Top-10 genes per vascular cell type

## 1. Imports & Gene List

In [ ]:
%matplotlib inline
import numpy as np, scipy.sparse as sp, anndata as ad, os, time
from collections import defaultdict
from scipy.stats import false_discovery_control
import csv, pandas as pd, numpy as np
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
import matplotlib.cm as cm
import scanpy as sc
import pandas as pd
from scipy import sparse

# ── Load gene list using user-specified columns ──────────────────────────────
df_genes = pd.read_csv(GENE_CSV_PATH)

# Validate required columns
for col in [DISEASE_COL, GENE_COL]:
    if col not in df_genes.columns:
        raise ValueError(f"Column '{col}' not found in CSV. "
                         f"Available columns: {df_genes.columns.tolist()}")

# Normalise column names for downstream use
if DISEASE_COL != 'disease' or GENE_COL != 'gene_name':
    df_genes = df_genes.rename(columns={DISEASE_COL: 'disease', GENE_COL: 'gene_name'})

print(df_genes.columns)
print(f"Loaded {len(df_genes)} entries | {df_genes['gene_name'].nunique()} unique genes")

display(df_genes.groupby('disease')['gene_name'].count().rename('n_genes').to_frame())


### Optional: add or remove genes

Edit the cells below to customise the gene list before running EWCE.
Leave both cells unchanged to use the v3 CSV as-is.

In [ ]:
# ── REMOVE genes (by gene symbol) ─────────────────────────────────────────────
remove_genes = []   # e.g. ["GENE1", "GENE2"]
if remove_genes:
    before = len(df_genes)
    df_genes = df_genes[~df_genes['gene_name'].isin(remove_genes)].reset_index(drop=True)
    print(f"Removed {before - len(df_genes)} rows for: {remove_genes}")
else:
    print("No genes removed.")

# Final summary
print(f"\nFinal gene list: {len(df_genes)} entries | "
      f"{df_genes['gene_name'].nunique()} unique genes | "
      f"{df_genes['disease'].nunique()} diseases")

## 2. Colour Palettes

In [ ]:
# ── Auto-generate disease & pathway colour palettes ─────────────────────────
diseases = sorted(df_genes['disease'].unique())

_dis_cmap = plt.get_cmap('tab20', len(diseases))
DISEASE_COLORS = {
    d: '#{:02X}{:02X}{:02X}'.format(*[int(x * 255) for x in _dis_cmap(i)[:3]])
    for i, d in enumerate(diseases)
}

# Keep vascular type colours fixed
VASC_COLORS = {"Endothelial": "#fcbe05", "Mural_Cell": "#A26DC2", "Fibroblast": "#5b844d"}
CT_COLORS = {
    "Astrocyte": "#F06719", "Neuron": "#08415C", "Ependymal_Cell": "#23767C",
    "Epithelial_Cell": "#155a66", "Microglia_Macrophage_T": "#DC143C",
    "Oligodendrocyte": "#00BFC4", "Endothelial": "#fcbe05",
    "Mural_Cell": "#A26DC2", "Fibroblast": "#5b844d", "OPC": "#0072B2",
}

# Preview colour strips
fig, axes = plt.subplots(1, 1, figsize=(14, 3))
for ax, pal, title in [(axes, DISEASE_COLORS, 'Disease colours')]:
    for i, (lbl, col) in enumerate(pal.items()):
        ax.barh(0, 1, left=i, color=col, edgecolor='white', linewidth=0.5)
        ax.text(i + 0.5, 0, lbl[:18], ha='center', va='center',
                fontsize=5, rotation=90, color='white')
    ax.set_xlim(0, len(pal))
    ax.axis('off')
    ax.set_title(title, fontsize=9, loc='left')

plt.tight_layout()
plt.show()

## 3. Configuration

In [ ]:
import numpy as np

# All paths and parameters come from the User Configuration cell above.
rng = np.random.default_rng(SEED)
os.makedirs(OUT, exist_ok=True)
print("Config OK. Output dir:", OUT)


## 4. Load AnnData

In [ ]:
print("Loading AnnData (backed mode) ...", flush=True)
t0    = time.time()
adata = sc.read_h5ad(H5AD_PATH, backed='r')
print(f"  shape={adata.shape}  ({time.time()-t0:.1f}s)")

# Validate obs columns
for col in [CELL_CLASS_COL, DONOR_COL]:
    if col not in adata.obs.columns:
        raise ValueError(f"Column '{col}' not found in adata.obs. "
                         f"Available columns: {adata.obs.columns.tolist()}")

obs       = adata.obs[[CELL_CLASS_COL, DONOR_COL]].copy()
var_names = adata.var_names.tolist()
gene2idx  = {g: i for i, g in enumerate(var_names)}
n_genes   = adata.n_vars
n_cells   = adata.n_obs
cell_types = obs[CELL_CLASS_COL].astype('category').cat.categories.tolist()
n_ct       = len(cell_types)
ct_labels  = obs[CELL_CLASS_COL].astype('category').cat.codes.values
ct_to_idx  = {ct: i for i, ct in enumerate(cell_types)}

donors       = obs[DONOR_COL].astype("category")
donor_labels = donors.cat.codes.values
print(f"Cell types ({n_ct}): {cell_types}")
print(f"Donors: {donors.nunique()}")


## 5. Pre-compute Library Sizes

In [ ]:
print("Computing per-cell library sizes ...", flush=True)
lib_sizes = np.zeros(n_cells, dtype=np.float64)
X = adata.X
for gs in range(0, n_genes, CHUNK):
    ge  = min(gs + CHUNK, n_genes)
    blk = X[:, gs:ge]
    lib_sizes += np.asarray(blk.sum(axis=1) if sp.issparse(blk)
                            else blk.sum(axis=1)).flatten()
    if gs % 10000 == 0:
        print(f"  {gs:>6}/{n_genes}", flush=True)
lib_sizes = np.maximum(lib_sizes, 1)
print(f"Done. min={lib_sizes.min():.0f}  max={lib_sizes.max():.0f}")

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(np.log10(lib_sizes), bins=80, color="#4DBBD5", edgecolor="none", alpha=0.8)
ax.set_xlabel("log₁₀(library size)"); ax.set_ylabel("cells")
ax.set_title("Per-cell library size distribution")
plt.tight_layout(); plt.show()

## 6. Donor-balanced CPM Means

In [ ]:
ct_donor_np = defaultdict(dict)
for i in range(n_cells):
    ct_donor_np[ct_labels[i]].setdefault(donor_labels[i], []).append(i)
ct_donor_np = {ci: {di: np.array(idxs) for di, idxs in dd.items()}
               for ci, dd in ct_donor_np.items()}
for ci in range(n_ct):
    nd = len(ct_donor_np.get(ci, {}))
    nc = sum(len(v) for v in ct_donor_np.get(ci, {}).values())
    print(f"  {cell_types[ci]}: {nd} donors, {nc} cells")

In [ ]:
# print(f"Computing donor-balanced CPM means ({n_genes} genes × {CHUNK}-gene chunks) ...")
# mean_expr = np.zeros((n_ct, n_genes), dtype=np.float32)
# for start in range(0, n_genes, CHUNK):
#     end   = min(start + CHUNK, n_genes)
#     block = X[:, start:end]
#     if sp.issparse(block): block = block.toarray()
#     cpm = block.astype(np.float64) / lib_sizes[:, None] * 1e6
#     for ci in range(n_ct):
#         dmeans = [cpm[idxs].mean(axis=0) for idxs in ct_donor_np.get(ci, {}).values()]
#         if dmeans:
#             mean_expr[ci, start:end] = np.mean(dmeans, axis=0).astype(np.float32)
#     if start % 10000 == 0:
#         print(f"  {start:>6}/{n_genes}", flush=True)
# print(f"Done. mean_expr: {mean_expr.shape}")

In [ ]:
import anndata as ad
import pandas as pd
import numpy as np
import scipy.sparse as sp

def _to_numeric_csr(X):
    """Convert matrix/layer to in-memory numeric scipy CSR."""
    if hasattr(X, "to_memory"):   # AnnData backed sparse dataset
        X = X.to_memory()

    if hasattr(X, "get"):         # cupy arrays
        X = X.get()

    if sp.issparse(X):
        return X.astype(np.float64).tocsr()

    X = np.asarray(X, dtype=np.float64)
    return sp.csr_matrix(X)


def pseudobulk_avg_cpm_by_celltype(
    adata,
    donor_key="donor",
    sample_key="sample",
    celltype_key="cell_type",
    count_layer=None,
    min_cells=10,
):
    X = adata.layers[count_layer] if count_layer is not None else adata.X
    X = _to_numeric_csr(X)

    obs = adata.obs.copy()

    for col in [donor_key, celltype_key]:
        if col not in obs.columns:
            raise ValueError(f"Missing required obs column: {col}")

    # donor x celltype cell counts
    cell_counts = (
        obs.groupby([donor_key, celltype_key], observed=True)
        .size()
        .rename("n_cells")
        .reset_index()
    )

    valid_groups = cell_counts[cell_counts["n_cells"] >= min_cells].copy()
    if valid_groups.empty:
        raise ValueError("No donor x cell_type groups passed min_cells filter.")

    valid_group_set = set(
        zip(valid_groups[donor_key].astype(str), valid_groups[celltype_key].astype(str))
    )

    keep_mask = np.array([
        (str(d), str(ct)) in valid_group_set
        for d, ct in zip(obs[donor_key], obs[celltype_key])
    ])

    obs_sub = obs.loc[keep_mask].copy()
    X_sub = X[keep_mask]

    obs_sub["pb_key"] = (
        obs_sub[donor_key].astype(str) + "||" + obs_sub[celltype_key].astype(str)
    )

    pb_keys = obs_sub["pb_key"].unique().tolist()
    pb_index = {k: i for i, k in enumerate(pb_keys)}

    row_idx = np.array([pb_index[k] for k in obs_sub["pb_key"]], dtype=int)
    col_idx = np.arange(obs_sub.shape[0], dtype=int)

    M = sp.csr_matrix(
        (np.ones(obs_sub.shape[0], dtype=np.float64), (row_idx, col_idx)),
        shape=(len(pb_keys), obs_sub.shape[0]),
    )

    # pseudobulk counts: donor x celltype
    pb_X = M @ X_sub

    pb_obs = pd.DataFrame({"pb_key": pb_keys})
    pb_obs[[donor_key, celltype_key]] = pb_obs["pb_key"].str.split(r"\|\|", expand=True)

    n_cells_map = {
        f"{row[donor_key]}||{row[celltype_key]}": row["n_cells"]
        for _, row in valid_groups.iterrows()
    }
    pb_obs["n_cells"] = pb_obs["pb_key"].map(n_cells_map)
    pb_obs.index = pb_obs["pb_key"]

    pb_adata = ad.AnnData(
        X=pb_X,
        obs=pb_obs,
        var=adata.var.copy(),
    )

    # CPM
    libsize = np.asarray(pb_adata.X.sum(axis=1)).ravel()
    X_dense = pb_adata.X.toarray() if sp.issparse(pb_adata.X) else np.asarray(pb_adata.X)

    cpm = np.zeros_like(X_dense, dtype=np.float64)
    nz = libsize > 0
    cpm[nz] = X_dense[nz] / libsize[nz, None] * 1e6

    donor_cpm = pd.DataFrame(cpm, index=pb_adata.obs_names, columns=pb_adata.var_names)
    donor_cpm[donor_key] = pb_adata.obs[donor_key].values
    donor_cpm[celltype_key] = pb_adata.obs[celltype_key].values
    donor_cpm["n_cells"] = pb_adata.obs["n_cells"].values

    avg_cpm = (
        donor_cpm
        .drop(columns=[donor_key, celltype_key, "n_cells"])
        .groupby(pb_adata.obs[celltype_key].values)
        .mean()
    )
    avg_cpm.index.name = celltype_key

    return avg_cpm, donor_cpm, pb_adata

In [ ]:
mean_expr, donor_cpm, pb_adata = pseudobulk_avg_cpm_by_celltype(
    adata,
    donor_key="individualID",
    sample_key="sampleID",
    celltype_key="Cell_class",
    min_cells=3,
)

## 7. Specificity Matrix

In [ ]:
mean_expr = mean_expr.to_numpy()
gene_total     = mean_expr.sum(axis=0)
expressed      = gene_total > 0
mean_expr_expr = mean_expr[:, expressed]
var_names_expr = [g for g, e in zip(var_names, expressed) if e]
gene2idx_expr  = {g: i for i, g in enumerate(var_names_expr)}
n_expr         = len(var_names_expr)

gene_sum = mean_expr_expr.sum(axis=0, keepdims=True)
spec_mat = mean_expr_expr / gene_sum      # (n_ct × n_expr)
spec_T   = spec_mat.T                     # (n_expr × n_ct)
print(f"Expressed genes: {n_expr} / {n_genes}")

df_spec = pd.DataFrame(spec_mat.T, index=var_names_expr, columns=cell_types)
df_spec.to_csv(f"{OUT}/ewce_specificity_matrix.csv")
print(f"Saved: {OUT}/ewce_specificity_matrix.csv")

In [ ]:
# QC: top-10 most specific genes per vascular cell type
fig, axes = plt.subplots(1, len(VASC_TYPES), figsize=(5*len(VASC_TYPES), 4))
for ax, ct in zip(axes, VASC_TYPES):
    ci    = ct_to_idx.get(ct)
    if ci is None: continue
    top10 = np.argsort(spec_T[:, ci])[-10:][::-1]
    genes = [var_names_expr[i] for i in top10]
    vals  = spec_T[top10, ci]
    ax.barh(genes[::-1], vals[::-1], color=VASC_COLORS.get(ct, "#999"))
    ax.set_title(f"{ct}\nTop-10 most specific genes", fontsize=9)
    ax.set_xlabel("Specificity"); ax.tick_params(labelsize=8)
plt.suptitle("Specificity matrix QC", fontsize=11, y=1.01)
plt.tight_layout(); plt.show()

## 8. Check Gene–Atlas Overlap

In [ ]:
disease_genes_all = df_genes['gene_name'].unique().tolist()
in_atlas     = [g for g in disease_genes_all if g in gene2idx_expr]
not_in_atlas = [g for g in disease_genes_all if g not in gene2idx_expr]
print(f"In atlas: {len(in_atlas)} / {len(disease_genes_all)}")
if not_in_atlas:
    missing_df = df_genes[df_genes['gene_name'].isin(not_in_atlas)][['disease','gene_name']].drop_duplicates()
    print(f"Not found ({len(not_in_atlas)}):")
    display(missing_df)

## 9. EWCE Bootstrap

In [ ]:
def ewce_test(gene_set, target_ct_idx, n_boot=N_BOOT):
    gene_idxs = np.array([gene2idx_expr[g] for g in gene_set if g in gene2idx_expr])
    if len(gene_idxs) == 0: return np.nan, np.nan, np.nan
    n_q  = len(gene_idxs)
    obs  = spec_T[gene_idxs, target_ct_idx].mean()
    null = np.array([spec_T[rng.integers(0, n_expr, n_q), target_ct_idx].mean()
                     for _ in range(n_boot)], dtype=np.float32)
    mu, sigma = null.mean(), null.std()
    ses = (obs - mu) / sigma if sigma > 0 else 0.0
    return float(obs), float(ses), float((null >= obs).sum() / n_boot)

all_ct_indices = {ct: ct_to_idx[ct] for ct in cell_types}
diseases_list  = df_genes['disease'].unique()
records        = []

for disease in diseases_list:
    gene_set_in = [g for g in df_genes[df_genes['disease'] == disease]['gene_name']
                   if g in gene2idx_expr]
    if len(gene_set_in) < 2:
        print(f"  Skip {disease}: {len(gene_set_in)} gene(s) in atlas"); continue
    for ct, ct_idx in all_ct_indices.items():
        obs_s, ses, pval = ewce_test(gene_set_in, ct_idx)
        records.append(dict(disease=disease, cell_type=ct,
                            n_genes=len(gene_set_in), obs_score=obs_s, ses=ses, p_value=pval))
    print(f"  Done: {disease} ({len(gene_set_in)} genes)", flush=True)

df_ewce = pd.DataFrame(records)
print(f"\nEWCE complete: {len(df_ewce)} rows")

## 10. FDR & Results

In [ ]:
for ct in cell_types:
    mask = df_ewce["cell_type"] == ct
    if mask.sum() == 0: continue
    df_ewce.loc[mask, "fdr"] = false_discovery_control(
        df_ewce.loc[mask, "p_value"].fillna(1.0).values, method='bh')

df_ewce = df_ewce.sort_values(["cell_type","ses"], ascending=[True, False])
df_ewce.to_csv(f"{OUT}/ewce_results.csv", index=False)
print(f"Saved: {OUT}/ewce_results.csv")

display(df_ewce[df_ewce["cell_type"].isin(VASC_TYPES)]
        [["disease","cell_type","n_genes","ses","fdr"]]
        .style.background_gradient(subset=["ses"], cmap="RdBu_r")
               .format({"ses": "{:.2f}", "fdr": "{:.3f}"}))

In [ ]:
import seaborn as sns

pivot_ses = df_ewce.pivot(index="disease", columns="cell_type", values="ses")
pivot_fdr = df_ewce.pivot(index="disease", columns="cell_type", values="fdr")
ct_order  = VASC_TYPES + [c for c in pivot_ses.columns if c not in VASC_TYPES]
pivot_ses = pivot_ses.reindex(columns=ct_order)
pivot_fdr = pivot_fdr.reindex(columns=ct_order)
vasc_cols = [c for c in VASC_TYPES if c in pivot_ses.columns]
pivot_ses["_vm"] = pivot_ses[vasc_cols].max(axis=1)
pivot_ses = pivot_ses.sort_values("_vm", ascending=False).drop(columns="_vm")
pivot_fdr = pivot_fdr.loc[pivot_ses.index]

fig, ax = plt.subplots(figsize=(len(ct_order)*0.9+2, len(pivot_ses)*0.55+2))
sns.heatmap(pivot_ses, cmap="RdBu_r", center=0, vmin=-4, vmax=16,
            linewidths=0.5, linecolor="#cccccc",
            cbar_kws={"label": "SES", "shrink": 0.6}, ax=ax)
for ri, dis in enumerate(pivot_ses.index):
    for ci, ct in enumerate(ct_order):
        fv = pivot_fdr.loc[dis, ct] if ct in pivot_fdr.columns else np.nan
        if pd.notna(fv) and fv < 0.05:
            ax.text(ci+0.5, ri+0.5, "*", ha="center", va="center",
                    fontsize=14, color="black", fontweight="bold")
ax.axvline(len(vasc_cols), color="black", linewidth=2, linestyle="--")
ax.set_title("EWCE Enrichment — All Diseases × All Cell Types  (* FDR < 0.05)", fontsize=11)
# ax.tick_params(axis='x', labelrotation=35)
plt.tight_layout()
fig.savefig(f"{OUT}/ewce_overview_all_celltypes.svg", bbox_inches="tight")
plt.show()

In [ ]:
# SES bar chart — vascular cell types only
df_vasc = df_ewce[df_ewce["cell_type"].isin(VASC_TYPES)].copy()
fig, axes = plt.subplots(1, len(VASC_TYPES),
                          figsize=(5*len(VASC_TYPES), max(4, len(diseases_list)*0.38+1)),
                          sharey=True)
for ax, ct in zip(axes, VASC_TYPES):
    sub    = df_vasc[df_vasc["cell_type"] == ct].sort_values("ses")
    colors = [DISEASE_COLORS.get(d, "#AAAAAA") for d in sub["disease"]]
    bars   = ax.barh(sub["disease"], sub["ses"], color=colors, edgecolor="none")
    for bar, sig in zip(bars, (sub["fdr"] < 0.05).tolist()):
        if sig:
            ax.text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2,
                    "*", va="center", fontsize=12, fontweight="bold")
    ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title(ct, fontsize=10, color=VASC_COLORS.get(ct, "#000"))
    ax.set_xlabel("SES"); ax.tick_params(labelsize=8)
plt.suptitle("EWCE — SES per vascular cell type  (* FDR < 0.05)", fontsize=11, y=1.01)
plt.tight_layout(); plt.show()

## 11. Top-10 Genes Per Vascular Cell Type

In [ ]:
vasc_ct_indices  = {ct: ct_to_idx[ct] for ct in VASC_TYPES if ct in ct_to_idx}
top_gene_records = []

for ct in VASC_TYPES:
    ct_idx = vasc_ct_indices.get(ct)
    if ct_idx is None: continue
    sig_dis = df_ewce[(df_ewce["cell_type"]==ct)&(df_ewce["fdr"]<0.05)]["disease"].tolist()
    if not sig_dis:
        sig_dis = df_ewce[df_ewce["cell_type"]==ct].nlargest(3,"ses")["disease"].tolist()
    pool    = df_genes[df_genes["disease"].isin(sig_dis)]["gene_name"].unique()
    pool_in = [g for g in pool if g in gene2idx_expr]
    if not pool_in: continue
    ranked = sorted({g: spec_T[gene2idx_expr[g], ct_idx] for g in pool_in}.items(),
                    key=lambda x: x[1], reverse=True)[:10]
    for rank, (gene, score) in enumerate(ranked, 1):
        row = df_genes[df_genes["gene_name"]==gene].iloc[0]
        top_gene_records.append(dict(
            cell_type=ct, rank=rank, gene_name=gene,
            disease=row["disease"], specificity_score=score))

df_top = pd.DataFrame(top_gene_records)
df_top.to_csv(f"{OUT}/ewce_top_genes.csv", index=False)
print(f"Saved: {OUT}/ewce_top_genes.csv")
display(df_top[["cell_type","rank","gene_name","disease","specificity_score"]]
        .style.background_gradient(subset=["specificity_score"], cmap="YlOrRd"))

In [ ]:
fig, axes = plt.subplots(1, len(VASC_TYPES), figsize=(5*len(VASC_TYPES), 5))
for ax, ct in zip(axes, VASC_TYPES):
    sub    = df_top[df_top["cell_type"]==ct].sort_values("rank")
    colors = [DISEASE_COLORS.get(d, "#AAAAAA") for d in sub["disease"]]
    ax.barh(sub["gene_name"][::-1], sub["specificity_score"][::-1],
            color=colors[::-1], edgecolor="none")
    ax.set_title(f"{ct}\nTop-10 EWCE genes", fontsize=9, color=VASC_COLORS.get(ct,"#000"))
    ax.set_xlabel("Specificity")
    ax.tick_params(labelsize=8)
#     for tick, pw in zip(ax.get_yticklabels(), sub["pathway_short"].tolist()[::-1]):
#         tick.set_color(PATHWAY_COLORS.get(pw, "#000"))
plt.suptitle("Top-10 EWCE genes  (bar=specificity; colour=disease; label colour=pathway)",
             fontsize=10, y=1.01)
plt.tight_layout(); plt.show()

## 12. Summary

Outputs saved → `../Results/`
- `ewce_specificity_matrix.csv`
- `ewce_results.csv`
- `ewce_top_genes.csv`

Proceed to **Step 3** notebook for pseudobulk heatmaps and dot plots.